Fasion MNIST Classifier

In [36]:
#bibliotheken
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Input, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import categorical_crossentropy


import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

2.20.0


1. Daten laden und verarbeiten

In [44]:
# Data importieren 
# Bilder 28x28 pxl NumPy Arrays, labels in range 0...9 
# Train Bilder: 60000, Test Bilder: 10000

fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels),(test_images, test_labels) = fashion_mnist.load_data()

print(train_labels[0])
print(train_images[0])

9
[[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   1   0   0  13  73   0
    0   1   4   0   0   0   0   1   1   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   3   0  36 136 127  62
   54   0   0   0   1   3   4   0   0   3]
 [  0   0   0   0   0   0   0   0   0   0   0   0   6   0 102 204 176 134
  144 123  23   0   0   0   0  12  10   0]
 [  0   0   0   0   0   0   0   0   0   0   0   0   0   0 155 236 207 178
  107 156 161 109  64  23  77 130  72  15]
 [  0   0   0   0   0   0   0   0   0   0   0   1   0  69 207 223 218 216
  216 163 127 121 122 146 141  88 172  66]
 [  0   0   0   0   0   0   0   0   0   1   1   1   0 200 232 

In [45]:
#Klassen Namen Array initialisieren
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(list(enumerate(class_names)))

[(0, 'T-shirt/top'), (1, 'Trouser'), (2, 'Pullover'), (3, 'Dress'), (4, 'Coat'), (5, 'Sandal'), (6, 'Shirt'), (7, 'Sneaker'), (8, 'Bag'), (9, 'Ankle boot')]


In [46]:
#Data type und shape
print(type(train_images))
print(train_images.shape)
len(train_labels)

<class 'numpy.ndarray'>
(60000, 28, 28)


60000

In [47]:
# Data normalisieren (von 0...255 Pixelwerte nach 0...1 Werte)
train_images = train_images/255.0
test_images = test_images/255.0

np.set_printoptions(suppress=True)  # !!! damit wir kein 0.00000000e+00 in print bekommen
print(train_images[0])

[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.00392157 0.    

2. Modellarchitektur bauen

In [48]:
#Erstellung eines Sequential-Modell
#!!! Input Layer mis erste sein !!!
#mit Flatten: 28x28 pxl to 1D Vektor

model = Sequential([    
                    Input(shape=(28,28)),
                    Flatten(),
                    Dense(128,activation="relu"),
                    Dense(10,activation='softmax')
                    ])

# model = tf.keras.Sequential([
    
#     tf.keras.layers.Flatten(input_shape=(28,28)),
#     tf.keras.layers.Dense(128,activation="relu"),
#     tf.keras.layers.Dense(10, activation="softmax")
# ])

In [49]:
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_7 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

3. Kompilieren

In [50]:
# Modell kompilieren
model.compile(Adam(),                              #logits sind rohe Werte 
              loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics = ['accuracy']
)

4. Training und Evaluation

In [ ]:
#training
model.fit(train_images, train_labels, epochs=10,batch_size=32, validation_split=0.2)

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8192 - loss: 0.5152 - val_accuracy: 0.8526 - val_loss: 0.4208
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8603 - loss: 0.3834 - val_accuracy: 0.8646 - val_loss: 0.3789
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8740 - loss: 0.3434 - val_accuracy: 0.8706 - val_loss: 0.3501
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8821 - loss: 0.3209 - val_accuracy: 0.8753 - val_loss: 0.3563
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8882 - loss: 0.2997 - val_accuracy: 0.8778 - val_loss: 0.3375
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.8950 - loss: 0.2842 - val_accuracy: 0.8842 - val_loss: 0.3290
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9000 - loss: 0.2725 - val_accuracy: 0.8714 - val_loss: 0.3590
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9034 - loss: 0.2595 - 

In [54]:
#evaluation
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print(f"Test accuracy: {test_acc:.4f}")

313/313 - 0s - 1ms/step - accuracy: 0.8790 - loss: 0.3419
Test accuracy: 0.8790


5. Make predictions

In [ ]:
# neues Modell
propability_model = Sequential([model, keras.layers.Softmax()])

In [56]:
#Vorhersage machen
pred = propability_model.predict(test_images)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [ ]:
#erste Vorhersage
pred[0]
#array von 10 Wahrscheinlichkeiten der Labels 0...9

array([0.08627381, 0.08626948, 0.08626989, 0.08626948, 0.08627007,
       0.08673114, 0.08626958, 0.09298753, 0.08627027, 0.21638872],
      dtype=float32)

In [58]:
#welches Label den höchste Konfidenzwert hat
np.argmax(pred[0])

np.int64(9)

In [59]:
#Entscheidung prüfen
test_labels[0]

np.uint8(9)

In [73]:
#Andere Vorhersage mit Realität verdleichen
n=11
pred[n]
print(f"Klassifizierung: {np.argmax(pred[n])}")
print(f"Original: {test_labels[n]}")

Klassifizierung: 5
Original: 5


6. Visualisierung. Confusion Matrix

In [79]:
# mit plot_model aus Keras
from tensorflow.keras.utils import plot_model

plot_model(model, to_file='Fashion_MNIST_model_diagram.png', show_shapes=True, show_layer_names=True)


You must install pydot (`pip install pydot`) for `plot_model` to work.
